# Module 4: Neo4j MCP
In this module we will install and use the [Neo4j MCP](https://neo4j.com/docs/mcp/current/) (Model Context Protocol).

Neo4j MCP gives AI assistants and LLM-powered tools direct, structured access to your Neo4j graph database. By implementing the Model Context Protocol (MCP), it acts as a bridge between any MCP-compatible client, such as Claude, Cursor, or VS Code with MCP support, and your Neo4j instance.

Neo4j MCP is intended for:

* Developers building or prototyping graph-backed AI applications who want to query Neo4j conversationally during development.

* Data scientists and analysts who want to explore graph data without deep cypher expertise.

* Platform and infrastructure teams deploying shared AI tooling that needs structured, auditable access to a Neo4j instance.

* AI application builders integrating Neo4j as a knowledge source or reasoning backend in multi-agent systems.

Neo4j MCP enables AI agents to:

* Explore your graph schema - discover node labels, relationship types, and property keys so the AI can reason about your data model without prior knowledge of it.

* Run Cypher queries - execute, read, and write queries against your database in response to natural language prompts.

* Inspect and analyze data - retrieve nodes, relationships, and paths to answer questions, generate summaries, or feed data to other workflows.

## Installation
To install Neo4j MCP, run the following cell:

In [ ]:
%%bash

curl -L -o /tmp/neo4j-mcp.tar.gz https://github.com/neo4j/mcp/releases/download/v1.5.2/neo4j-mcp_Linux_x86_64.tar.gz
mkdir -p /tmp/neo4j-mcp
tar -xzf /tmp/neo4j-mcp.tar.gz -C /tmp/neo4j-mcp
chmod +x /tmp/neo4j-mcp/neo4j-mcp
sudo mv /tmp/neo4j-mcp/neo4j-mcp /usr/local/bin/
neo4j-mcp -v
set -a
source /workspaces/from_zero_to_hero/.env
set +a
mkdir -p /workspaces/from_zero_to_hero/.vscode
cat > /workspaces/from_zero_to_hero/.vscode/mcp.json <<EOF
{
  "servers": {
    "neo4j": {
      "type": "stdio",
      "command": "neo4j-mcp",
      "env": {
        "NEO4J_URI": "${NEO4J_URI}",
        "NEO4J_USERNAME": "${NEO4J_USERNAME}",
        "NEO4J_PASSWORD": "${NEO4J_PASSWORD}",
        "NEO4J_DATABASE": "${NEO4J_DATABASE}",
        "NEO4J_TELEMETRY": "false"
      }
    }
  }
}
EOF

Once the previous cell has finished running you should have a new folder called `.vscode` in your project directory. Inside that folder there should be a file called `mcp.json`. Open the file and verify that it looks something like this but with your Aura Free values for the environment variables:

```json
{
  "servers": {
    "neo4j": {
      "type": "stdio",
      "command": "neo4j-mcp",
      "env": {
        "NEO4J_URI": "neo4j+s://123456.databases.neo4j.io",
        "NEO4J_USERNAME": "123456",
        "NEO4J_PASSWORD": "123456",
        "NEO4J_DATABASE": "123456",
        "NEO4J_TELEMETRY": "false"
      }
    }
  }
}
```

Just beneath the servers section there is a start button. Click that to start the Neo4j MCP server. If it doesn't start successfully, try starting it again. Once it is running you should see the status turning to "running" and there should be three tools available.

# Tools
These are the tools that Neo4j MCP exposes:

|Tool|ReadOnly|Purpose|Notes
|:---|:---|:---|:---
|get-schema|Yes|Introspect labels, relationship types, property keys|Provide valuable context to the client LLMs.
|read-cypher|Yes|Execute arbitrary Cypher® (read mode)|Rejects writes, schema/admin operations, and PROFILE queries.
|write-cypher|No|Execute arbitrary Cypher (write mode)|Caution: LLM-generated queries can cause harm. Use only in development environments. Disabled if NEO4J_READ_ONLY=true.
|list-gds-procedures|Yes|List GDS procedures available in the instance|Help the client LLM to have better visibility on available GDS procedures.

The list-gds-procedures tool is not available in Aura Free since it does not allow installing the GDS library, but if you have a different Neo4j instance with GDS installed you can use that to get a list of available GDS procedures.

# Use the Neo4j MCP tools in a client
Now that you have the Neo4j MCP server running, you can use the tools it exposes in VS Code. 

Open the chat pane in VS Code and write a message in natural language asking to get the schema of the graph.

You'll be asked to allow the Neo4j MCP server to use the get-schema tool. Allow it and you should see a response with the schema of the graph. You can then ask follow up questions about the data and allow the server to use the read-cypher tool to get answers from the graph.

### Some example questions and actions to try out:
* "What persons are skilled in python?"
* "Who has the greatest overlap in skills with Miguel Santos?"
* "Find the shortest path between the most senior data engineer and the most junior data engineer."
* Try creating a new relationship between a person and a skill. For example, "Make Monica Garcia skilled in graph databases". Be cautious with this one as it will modify the data in your database.
